# Notebook 07 — Report Export Summary

Report-ready exports are separated into Core RQ2, five-model default benchmark, boosting Threshold Sensitivity Analysis, final frozen-threshold test results, and model interpretation.


## 1. Imports and explicit report scopes


In [27]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from PIL import Image

CORE_MODEL_ORDER = ["Logistic Regression", "Random Forest", "XGBoost"]
DEFAULT_BENCHMARK_ORDER = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "CatBoost"]
BOOSTING_MODEL_ORDER = ["XGBoost", "LightGBM", "CatBoost"]
RECALL_TARGET = 0.85
REPORT_PREFIXES = ("report_summary_", "report_table_", "report_figure_manifest")


## 2. Paths and export helpers


In [28]:
def find_source_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if candidate.name == "Source":
            return candidate
        nested = candidate / "Source"
        if nested.exists() and nested.is_dir():
            return nested.resolve()
    raise FileNotFoundError("Could not locate Source.")


CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()
SOURCE_ROOT = find_source_root(CURRENT_WORKING_DIRECTORY)
REPO_ROOT = SOURCE_ROOT.parent
TARGET_NOTEBOOK_PATH = SOURCE_ROOT / "notebooks" / "07_Report_Export_Summary.ipynb"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Source.src.path import DATA_REPORTS_DIR, FIGURES_DIR

DATA_REPORTS_DIR = Path(DATA_REPORTS_DIR)
FIGURES_DIR = Path(FIGURES_DIR)


def load_csv(filename: str) -> pd.DataFrame:
    path = DATA_REPORTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing report input: {path}")
    frame = pd.read_csv(path)
    if frame.empty:
        raise ValueError(f"Report input is empty: {path}")
    print(f"Loaded {filename}: shape={frame.shape}")
    return frame


def save_table(df: pd.DataFrame, filename: str, title: str) -> Path:
    if not filename.startswith(REPORT_PREFIXES):
        raise ValueError(f"Notebook 07 may only create report-prefixed outputs: {filename}")
    path = DATA_REPORTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"{title}: shape={df.shape}; saved={path}")
    display(df)
    return path


def save_text(content: str, filename: str, title: str) -> Path:
    if not filename.startswith(REPORT_PREFIXES):
        raise ValueError(f"Notebook 07 may only create report-prefixed outputs: {filename}")
    path = DATA_REPORTS_DIR / filename
    path.write_text(content, encoding="utf-8")
    print(f"{title}: saved={path}")
    return path


def assert_scope(df: pd.DataFrame, expected: list[str], title: str) -> None:
    actual = df["Model"].astype(str).tolist()
    if actual != expected:
        raise ValueError(f"{title} scope mismatch: {actual}")


def metric_leader(df: pd.DataFrame, metric: str) -> pd.Series:
    return df.sort_values([metric, "F1", "Recall"], ascending=False).iloc[0]


## 3. Load approved comparison and interpretation outputs


In [29]:
core_research = load_csv("core_research_default_threshold_validation_comparison.csv")
five_model_benchmark = load_csv("five_model_default_threshold_validation_benchmark.csv")
boosting_threshold_combined = load_csv("boosting_threshold_sensitivity_validation_combined.csv")
boosting_selected_validation = load_csv("boosting_selected_threshold_validation_comparison.csv")
boosting_selected_test = load_csv("boosting_selected_threshold_test_comparison.csv")
boosting_delta = load_csv("boosting_default_vs_selected_validation_delta.csv")

logistic_interpretation = load_csv("model_interpretation_logistic_coefficients_summary.csv")
rf_interpretation = load_csv("model_interpretation_random_forest_top_features.csv")
xgb_gain_interpretation = load_csv("model_interpretation_xgboost_top_features.csv")
lightgbm_interpretation = load_csv("model_interpretation_lightgbm_top_features.csv")
catboost_interpretation = load_csv("model_interpretation_catboost_top_features.csv")
xgb_shap_interpretation = load_csv("model_interpretation_xgboost_shap_mean_abs.csv")
rank_overlap = load_csv("model_interpretation_rank_overlap.csv")

expected_tenure_labels = {'Tenure': 'Tenure (Months)', 'tenure_group': 'Tenure Group'}
for source_name, frame in [('XGBoost gain', xgb_gain_interpretation), ('XGBoost SHAP', xgb_shap_interpretation), ('rank overlap', rank_overlap)]:
    labels = frame.set_index('Feature_Code')['Feature_Display_Name'].to_dict()
    for feature_code, expected_label in expected_tenure_labels.items():
        if feature_code in labels and labels[feature_code] != expected_label:
            raise ValueError(f'{source_name} has invalid label for {feature_code}: {labels[feature_code]}')
if rank_overlap['Feature_Code'].duplicated().any():
    raise ValueError('Cross-method rank overlap contains duplicate raw feature drivers.')

assert_scope(core_research, CORE_MODEL_ORDER, "Core RQ2")
assert_scope(five_model_benchmark, DEFAULT_BENCHMARK_ORDER, "Five-model benchmark")
assert_scope(boosting_selected_validation, BOOSTING_MODEL_ORDER, "Boosting selected validation")
assert_scope(boosting_selected_test, BOOSTING_MODEL_ORDER, "Boosting final test")
assert_scope(boosting_delta, BOOSTING_MODEL_ORDER, "Boosting delta")
if len(boosting_threshold_combined) != 273 or set(boosting_threshold_combined["Model"]) != set(BOOSTING_MODEL_ORDER):
    raise ValueError("Boosting threshold sensitivity must contain 273 rows and three boosting models.")


Loaded core_research_default_threshold_validation_comparison.csv: shape=(3, 20)
Loaded five_model_default_threshold_validation_benchmark.csv: shape=(5, 20)
Loaded boosting_threshold_sensitivity_validation_combined.csv: shape=(273, 20)
Loaded boosting_selected_threshold_validation_comparison.csv: shape=(3, 20)
Loaded boosting_selected_threshold_test_comparison.csv: shape=(3, 20)
Loaded boosting_default_vs_selected_validation_delta.csv: shape=(3, 29)
Loaded model_interpretation_logistic_coefficients_summary.csv: shape=(26, 10)
Loaded model_interpretation_random_forest_top_features.csv: shape=(15, 8)
Loaded model_interpretation_xgboost_top_features.csv: shape=(15, 8)
Loaded model_interpretation_lightgbm_top_features.csv: shape=(15, 8)
Loaded model_interpretation_catboost_top_features.csv: shape=(15, 8)
Loaded model_interpretation_xgboost_shap_mean_abs.csv: shape=(18, 8)
Loaded model_interpretation_rank_overlap.csv: shape=(20, 7)


## 4. A — Core Research Comparison (RQ2)


In [30]:
report_core_columns = ["Model", "Dataset", "Threshold", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "TP", "FP", "TN", "FN", "FPR", "FNR"]
report_summary_core = core_research[report_core_columns].copy()
if not (report_summary_core["Threshold"] == 0.50).all() or not (report_summary_core["Dataset"] == "validation").all():
    raise ValueError("RQ2 must use validation at default threshold 0.50.")
save_table(report_summary_core, "report_summary_core_research_comparison.csv", "Core RQ2 comparison")


Core RQ2 comparison: shape=(3, 15); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_core_research_comparison.csv


,Model,Dataset,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TP,FP,TN,FN,FPR,FNR
0,Logistic Regression,validation,0.5,0.759470,0.530660,0.803571,0.639205,0.854773,0.665839,225,199,577,55,0.256443,0.196429
1,Random Forest,validation,0.5,0.790720,0.629956,0.510714,0.564103,0.845214,0.645568,143,84,692,137,0.108247,0.489286
2,XGBoost,validation,0.5,0.761364,0.533493,0.796429,0.638968,0.854375,0.665582,223,195,581,57,0.251289,0.203571


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_core_research_comparison.csv')

## 5. B — Five-Model Default Benchmark


In [31]:
report_summary_five_model = five_model_benchmark[report_core_columns].copy()
if not (report_summary_five_model["Threshold"] == 0.50).all() or not (report_summary_five_model["Dataset"] == "validation").all():
    raise ValueError("Five-model benchmark must use validation at default threshold 0.50.")
save_table(report_summary_five_model, "report_summary_five_model_default_benchmark.csv", "Five-model default benchmark")


Five-model default benchmark: shape=(5, 15); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_five_model_default_benchmark.csv


,Model,Dataset,Threshold,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TP,FP,TN,FN,FPR,FNR
0,Logistic Regression,validation,0.5,0.759470,0.530660,0.803571,0.639205,0.854773,0.665839,225,199,577,55,0.256443,0.196429
1,Random Forest,validation,0.5,0.790720,0.629956,0.510714,0.564103,0.845214,0.645568,143,84,692,137,0.108247,0.489286
2,XGBoost,validation,0.5,0.761364,0.533493,0.796429,0.638968,0.854375,0.665582,223,195,581,57,0.251289,0.203571
3,LightGBM,validation,0.5,0.779356,0.562334,0.757143,0.645358,0.848523,0.651637,212,165,611,68,0.212629,0.242857
4,CatBoost,validation,0.5,0.764205,0.538272,0.778571,0.636496,0.850150,0.655526,218,187,589,62,0.240979,0.221429


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_five_model_default_benchmark.csv')

## 6. C — Boosting Threshold Sensitivity Analysis


In [32]:
threshold_columns = [
    "Model", "Default_Threshold", "Selected_Threshold", "Validation_Recall_Target", "Meets_Recall_Target",
    "Default_Accuracy", "Selected_Accuracy", "Delta_Accuracy",
    "Default_Precision", "Selected_Precision", "Delta_Precision",
    "Default_Recall", "Selected_Recall", "Delta_Recall",
    "Default_F1", "Selected_F1", "Delta_F1",
    "Default_TP", "Selected_TP", "Delta_TP", "Default_FP", "Selected_FP", "Delta_FP",
    "Default_TN", "Selected_TN", "Delta_TN", "Default_FN", "Selected_FN", "Delta_FN",
]
report_summary_boosting_sensitivity = boosting_delta[threshold_columns].copy()
validation_target_lookup = dict(zip(boosting_selected_validation["Model"], boosting_selected_validation["Recall"] >= RECALL_TARGET))
report_summary_boosting_sensitivity["Meets_Recall_Target"] = report_summary_boosting_sensitivity["Model"].map(validation_target_lookup)
if not report_summary_boosting_sensitivity["Meets_Recall_Target"].all():
    raise ValueError("Recall target status must be met and determined from validation.")
save_table(report_summary_boosting_sensitivity, "report_summary_boosting_threshold_sensitivity.csv", "Boosting Threshold Sensitivity Analysis summary")


Boosting Threshold Sensitivity Analysis summary: shape=(3, 29); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_boosting_threshold_sensitivity.csv


,Model,Default_Threshold,Selected_Threshold,Validation_Recall_Target,Meets_Recall_Target,Default_Accuracy,Selected_Accuracy,Delta_Accuracy,Default_Precision,Selected_Precision,...,Delta_TP,Default_FP,Selected_FP,Delta_FP,Default_TN,Selected_TN,Delta_TN,Default_FN,Selected_FN,Delta_FN
0,XGBoost,0.5,0.38,0.85,True,0.761364,0.731061,-0.030303,0.533493,0.495833,...,15,195,242,47,581,534,-47,57,42,-15
1,LightGBM,0.5,0.27,0.85,True,0.779356,0.724432,-0.054924,0.562334,0.488798,...,28,165,251,86,611,525,-86,68,40,-28
2,CatBoost,0.5,0.38,0.85,True,0.764205,0.741477,-0.022727,0.538272,0.507431,...,21,187,232,45,589,544,-45,62,41,-21


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_boosting_threshold_sensitivity.csv')

## 7. D — Final Boosting Test Results


In [33]:
report_final_columns = ["Model", "Dataset", "Threshold", "Threshold_Source", "Accuracy", "Precision", "Recall", "F1", "ROC_AUC", "PR_AUC", "TP", "FP", "TN", "FN", "FPR", "FNR"]
report_summary_boosting_final = boosting_selected_test[report_final_columns].copy()
if not (report_summary_boosting_final["Dataset"] == "test").all() or not (report_summary_boosting_final["Threshold_Source"] == "validation").all():
    raise ValueError("Final boosting test thresholds must be frozen from validation.")
validation_thresholds = dict(zip(boosting_selected_validation["Model"], boosting_selected_validation["Threshold"]))
for _, row in report_summary_boosting_final.iterrows():
    if abs(row["Threshold"] - validation_thresholds[row["Model"]]) > 1e-12:
        raise ValueError(f"Frozen test threshold mismatch for {row['Model']}")
save_table(report_summary_boosting_final, "report_summary_boosting_final_test.csv", "Final boosting test results")


Final boosting test results: shape=(3, 16); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_boosting_final_test.csv


,Model,Dataset,Threshold,Threshold_Source,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC,TP,FP,TN,FN,FPR,FNR
0,XGBoost,test,0.38,validation,0.721854,0.486974,0.864769,0.623077,0.841160,0.653840,243,256,520,38,0.329897,0.135231
1,LightGBM,test,0.27,validation,0.703879,0.469349,0.871886,0.610212,0.831690,0.639849,245,277,499,36,0.356959,0.128114
2,CatBoost,test,0.38,validation,0.720908,0.486166,0.875445,0.625159,0.846108,0.663823,246,260,516,35,0.335052,0.124555


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_boosting_final_test.csv')

## 8. E — Model Interpretation Methods


In [34]:
interpretation_methods = pd.DataFrame([
    {"Model": "Logistic Regression", "Method": "Coefficients", "Unit": "logistic-model coefficient", "Detailed_SHAP": False, "Source_Output": "model_interpretation_logistic_coefficients_summary.csv"},
    {"Model": "Random Forest", "Method": "Impurity importance", "Unit": "impurity importance", "Detailed_SHAP": False, "Source_Output": "model_interpretation_random_forest_top_features.csv"},
    {"Model": "XGBoost", "Method": "Gain importance", "Unit": "mean gain", "Detailed_SHAP": False, "Source_Output": "model_interpretation_xgboost_top_features.csv"},
    {"Model": "LightGBM", "Method": "Gain importance", "Unit": "gain importance", "Detailed_SHAP": False, "Source_Output": "model_interpretation_lightgbm_top_features.csv"},
    {"Model": "CatBoost", "Method": "FeatureImportance", "Unit": "CatBoost native FeatureImportance", "Detailed_SHAP": False, "Source_Output": "model_interpretation_catboost_top_features.csv"},
    {"Model": "XGBoost", "Method": "SHAP", "Unit": "raw model margin / log-odds for Churn = 1", "Detailed_SHAP": True, "Source_Output": "model_interpretation_xgboost_shap_mean_abs.csv"},
])
interpretation_methods["Cross_Method_Comparison_Rule"] = "Compare feature ranks/overlap only; do not compare raw magnitudes."
save_table(interpretation_methods, "report_summary_model_interpretation_methods.csv", "Model interpretation methods")

report_summary_top_drivers = rank_overlap.head(12)[["Feature_Code", "Feature_Display_Name", "Driver_Group", "Method_Count", "Mean_Rank", "Methods", "Comparison_Basis"]].copy()
report_summary_top_drivers["Evidence_Statement"] = "Predictive association across model interpretation methods; not a causal effect."
save_table(report_summary_top_drivers, "report_summary_top_churn_drivers.csv", "Cross-method top churn drivers")
save_text(report_summary_top_drivers.to_latex(index=False, float_format="%.4f"), "report_table_top_churn_drivers_latex.tex", "Top churn drivers LaTeX table")


Model interpretation methods: shape=(6, 6); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_model_interpretation_methods.csv


,Model,Method,Unit,Detailed_SHAP,Source_Output,Cross_Method_Comparison_Rule
0,Logistic Regression,Coefficients,logistic-model coefficient,False,model_interpretation_logistic_coefficients_sum...,Compare feature ranks/overlap only; do not com...
1,Random Forest,Impurity importance,impurity importance,False,model_interpretation_random_forest_top_feature...,Compare feature ranks/overlap only; do not com...
2,XGBoost,Gain importance,mean gain,False,model_interpretation_xgboost_top_features.csv,Compare feature ranks/overlap only; do not com...
3,LightGBM,Gain importance,gain importance,False,model_interpretation_lightgbm_top_features.csv,Compare feature ranks/overlap only; do not com...
4,CatBoost,FeatureImportance,CatBoost native FeatureImportance,False,model_interpretation_catboost_top_features.csv,Compare feature ranks/overlap only; do not com...
5,XGBoost,SHAP,raw model margin / log-odds for Churn = 1,True,model_interpretation_xgboost_shap_mean_abs.csv,Compare feature ranks/overlap only; do not com...


Cross-method top churn drivers: shape=(12, 8); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_top_churn_drivers.csv


,Feature_Code,Feature_Display_Name,Driver_Group,Method_Count,Mean_Rank,Methods,Comparison_Basis,Evidence_Statement
0,Tenure,Tenure (Months),Tenure,6,4.833333,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
1,MonthlyCharges,Monthly Charges,Monthly charges / total charges,6,5.000000,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
2,Contract,Contract Type,Contract type,5,2.000000,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
3,TotalCharges,Total Charges,Monthly charges / total charges,5,2.800000,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
4,Dependents,Dependents,Other customer or service profile,5,3.400000,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
5,monthly_charges_ratio,Monthly Charges Ratio,Monthly charges / total charges,4,5.000000,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
6,InternetService,Internet Service,Internet service,4,7.000000,CatBoost FeatureImportance; Random Forest impu...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
7,PaymentMethod,Payment Method,Payment method,4,7.000000,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
8,service_diversity,Service Diversity Score,Service diversity,4,9.250000,CatBoost FeatureImportance; LightGBM gain impo...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...
9,tenure_group,Tenure Group,Tenure,3,6.666667,CatBoost FeatureImportance; Random Forest impu...,Top-10 rank presence only; raw magnitudes are ...,Predictive association across model interpreta...


Top churn drivers LaTeX table: saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_table_top_churn_drivers_latex.tex


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_table_top_churn_drivers_latex.tex')

## 9. F — Metric-Specific Recommendations


In [35]:
recommendation_rows = []
for scope, frame, metrics, basis in [
    ("Core RQ2", report_summary_core, ["Recall", "F1", "ROC_AUC"], "validation, threshold 0.50"),
    ("Five-model default benchmark", report_summary_five_model, ["Recall", "F1", "PR_AUC"], "validation, threshold 0.50"),
    ("Boosting final test", report_summary_boosting_final, ["Recall", "F1", "PR_AUC"], "test, threshold frozen from validation"),
]:
    for metric in metrics:
        leader = metric_leader(frame, metric)
        recommendation_rows.append({
            "Scope": scope, "Metric": metric, "Leading_Model": leader["Model"], "Metric_Value": leader[metric],
            "Evaluation_Basis": basis,
            "Interpretation": f"Highest {metric} within this specific scope; not a universal best-model claim.",
        })
report_metric_recommendations = pd.DataFrame(recommendation_rows)
save_table(report_metric_recommendations, "report_summary_metric_specific_recommendations.csv", "Metric-specific model findings")

business_recommendations = pd.DataFrame([
    {"Recommendation": "Prioritize contract and early-tenure retention review", "Predictive_Evidence": "Contract type and tenure rank consistently across interpretation methods.", "Caveat": "Predictive association; no causal claim."},
    {"Recommendation": "Review high-charge customer experience", "Predictive_Evidence": "Monthly and total charges appear among cross-method predictive drivers.", "Caveat": "Predictive association; no causal claim."},
    {"Recommendation": "Use support, security and service signals for targeted screening", "Predictive_Evidence": "Support/security and internet-service variables contribute to churn prediction.", "Caveat": "Interventions require separate causal evaluation."},
    {"Recommendation": "Select an operating model by the metric required", "Predictive_Evidence": "Recall, F1 and PR_AUC leaders differ by scope.", "Caveat": "No model is declared universally best."},
])
save_table(business_recommendations, "report_summary_business_recommendations.csv", "Business recommendations with evidence caveats")


Metric-specific model findings: shape=(9, 6); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_metric_specific_recommendations.csv


,Scope,Metric,Leading_Model,Metric_Value,Evaluation_Basis,Interpretation
0,Core RQ2,Recall,Logistic Regression,0.803571,"validation, threshold 0.50",Highest Recall within this specific scope; not...
1,Core RQ2,F1,Logistic Regression,0.639205,"validation, threshold 0.50",Highest F1 within this specific scope; not a u...
2,Core RQ2,ROC_AUC,Logistic Regression,0.854773,"validation, threshold 0.50",Highest ROC_AUC within this specific scope; no...
3,Five-model default benchmark,Recall,Logistic Regression,0.803571,"validation, threshold 0.50",Highest Recall within this specific scope; not...
4,Five-model default benchmark,F1,LightGBM,0.645358,"validation, threshold 0.50",Highest F1 within this specific scope; not a u...
5,Five-model default benchmark,PR_AUC,Logistic Regression,0.665839,"validation, threshold 0.50",Highest PR_AUC within this specific scope; not...
6,Boosting final test,Recall,CatBoost,0.875445,"test, threshold frozen from validation",Highest Recall within this specific scope; not...
7,Boosting final test,F1,CatBoost,0.625159,"test, threshold frozen from validation",Highest F1 within this specific scope; not a u...
8,Boosting final test,PR_AUC,CatBoost,0.663823,"test, threshold frozen from validation",Highest PR_AUC within this specific scope; not...


Business recommendations with evidence caveats: shape=(4, 3); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_business_recommendations.csv


,Recommendation,Predictive_Evidence,Caveat
0,Prioritize contract and early-tenure retention...,Contract type and tenure rank consistently acr...,Predictive association; no causal claim.
1,Review high-charge customer experience,Monthly and total charges appear among cross-m...,Predictive association; no causal claim.
2,"Use support, security and service signals for ...",Support/security and internet-service variable...,Interventions require separate causal evaluation.
3,Select an operating model by the metric required,"Recall, F1 and PR_AUC leaders differ by scope.",No model is declared universally best.


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_business_recommendations.csv')

## 10. Approved 300-DPI Figure Manifest


In [36]:
figure_manifest_specs = [
    ("eda_01_churn_distribution.png", "Notebook 01 EDA", "EDA overview", "Overall churn distribution."),
    ("eda_02_churn_rate_by_contract.png", "Notebook 01 EDA", "EDA segment analysis", "Churn rate by contract type."),
    ("eda_03_churn_rate_by_payment_method.png", "Notebook 01 EDA", "EDA segment analysis", "Churn rate by payment method."),
    ("eda_04_churn_rate_by_internet_service.png", "Notebook 01 EDA", "EDA segment analysis", "Churn rate by internet service."),
    ("eda_14_top_risk_segments.png", "Notebook 01 EDA", "EDA risk summary", "Top predictive-risk segments."),
    ("model_comparison_01_five_model_default_threshold_validation_metrics.png", "Notebook 05 Model Comparison", "Five-model default benchmark", "Validation metrics for five models at threshold 0.50."),
    ("model_comparison_02_boosting_selected_threshold_test_metrics.png", "Notebook 05 Model Comparison", "Final boosting test results", "Test metrics at validation-frozen thresholds."),
    ("model_comparison_08_boosting_threshold_sensitivity_recall_validation.png", "Notebook 05 Model Comparison", "Threshold Sensitivity Analysis", "Validation Recall curves and selected thresholds for three boosting models."),
    ("model_interpretation_01_logistic_top_coefficients.png", "Notebook 06 Interpretation", "Model-specific interpretation", "Logistic Regression coefficients."),
    ("model_interpretation_02_random_forest_top_importance.png", "Notebook 06 Interpretation", "Model-specific interpretation", "Random Forest impurity importance."),
    ("model_interpretation_03_xgboost_top_importance.png", "Notebook 06 Interpretation", "Model-specific interpretation", "XGBoost gain importance."),
    ("model_interpretation_08_lightgbm_top_importance.png", "Notebook 06 Interpretation", "Model-specific interpretation", "LightGBM gain importance."),
    ("model_interpretation_09_catboost_top_importance.png", "Notebook 06 Interpretation", "Model-specific interpretation", "CatBoost FeatureImportance."),
    ("model_interpretation_04_xgboost_shap_summary_bar.png", "Notebook 06 Interpretation", "XGBoost SHAP", "Mean absolute SHAP on raw margin for Churn = 1."),
    ("model_interpretation_05_xgboost_shap_beeswarm.png", "Notebook 06 Interpretation", "XGBoost SHAP", "SHAP beeswarm on validation sample for Churn = 1."),
    ("model_interpretation_06_xgboost_shap_dependence_top_feature.png", "Notebook 06 Interpretation", "XGBoost SHAP", "Dependence plot for the top SHAP feature."),
    ("model_interpretation_07_xgboost_shap_waterfall_high_risk_customer.png", "Notebook 06 Interpretation", "XGBoost SHAP", "Local raw-margin SHAP explanation for one validation customer."),
]

manifest_rows = []
for filename, notebook_source, section, caption in figure_manifest_specs:
    path = FIGURES_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Approved manifest figure is missing: {path}")
    with Image.open(path) as image:
        dpi = image.info.get("dpi", (0.0, 0.0))
        dpi_x, dpi_y = float(dpi[0]), float(dpi[1])
        width, height = image.size
    if dpi_x < 299.0 or dpi_y < 299.0:
        raise ValueError(f"Manifest figure is below 300 DPI: {filename}, dpi={dpi}")
    manifest_rows.append({
        "Figure filename": filename, "Notebook source": notebook_source,
        "Suggested report section": section, "Short caption": caption,
        "DPI_X": dpi_x, "DPI_Y": dpi_y, "Width_Pixels": width, "Height_Pixels": height,
        "File exists": True, "Path": path,
    })
figure_manifest = pd.DataFrame(manifest_rows)
save_table(figure_manifest, "report_figure_manifest.csv", "Approved 300-DPI report figure manifest")


Approved 300-DPI report figure manifest: shape=(17, 10); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_figure_manifest.csv


,Figure filename,Notebook source,Suggested report section,Short caption,DPI_X,DPI_Y,Width_Pixels,Height_Pixels,File exists,Path
0,eda_01_churn_distribution.png,Notebook 01 EDA,EDA overview,Overall churn distribution.,299.9994,299.9994,2050,1446,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
1,eda_02_churn_rate_by_contract.png,Notebook 01 EDA,EDA segment analysis,Churn rate by contract type.,299.9994,299.9994,3252,1747,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
2,eda_03_churn_rate_by_payment_method.png,Notebook 01 EDA,EDA segment analysis,Churn rate by payment method.,299.9994,299.9994,3252,1746,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
3,eda_04_churn_rate_by_internet_service.png,Notebook 01 EDA,EDA segment analysis,Churn rate by internet service.,299.9994,299.9994,3252,1746,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
4,eda_14_top_risk_segments.png,Notebook 01 EDA,EDA risk summary,Top predictive-risk segments.,299.9994,299.9994,3548,2348,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
5,model_comparison_01_five_model_default_thresho...,Notebook 05 Model Comparison,Five-model default benchmark,Validation metrics for five models at threshol...,299.9994,299.9994,3567,1914,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
6,model_comparison_02_boosting_selected_threshol...,Notebook 05 Model Comparison,Final boosting test results,Test metrics at validation-frozen thresholds.,299.9994,299.9994,3567,1914,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
7,model_comparison_08_boosting_threshold_sensiti...,Notebook 05 Model Comparison,Threshold Sensitivity Analysis,Validation Recall curves and selected threshol...,299.9994,299.9994,3117,1914,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
8,model_interpretation_01_logistic_top_coefficie...,Notebook 06 Interpretation,Model-specific interpretation,Logistic Regression coefficients.,299.9994,299.9994,2948,1972,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...
9,model_interpretation_02_random_forest_top_impo...,Notebook 06 Interpretation,Model-specific interpretation,Random Forest impurity importance.,299.9994,299.9994,2948,1972,True,D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_G...


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_figure_manifest.csv')

## 11. Explicit deprecation exclusions


In [37]:
deprecated_outputs = pd.DataFrame({"Deprecated_Output_Excluded": [
    "core_model_selected_threshold_validation_comparison.csv",
    "core_model_selected_threshold_test_comparison.csv",
    "core_model_threshold_sensitivity_combined.csv",
    "report_summary_model_comparison.csv",
    "report_summary_threshold_selection.csv",
    "model_comparison_01_selected_test_metrics_bar.png",
    "model_comparison_02_default_validation_metrics_bar.png",
    "model_comparison_03_selected_validation_metrics_bar.png",
    "model_comparison_04_selected_thresholds.png",
    "model_comparison_05_validation_vs_test_recall_f1.png",
    "model_comparison_06_precision_recall_tradeoff_test.png",
    "model_comparison_07_threshold_sensitivity_recall.png",
]})
deprecated_outputs["Reason"] = "Historical scope; excluded from approved mappings and manifest without deleting the file."
save_table(deprecated_outputs, "report_summary_deprecated_outputs_excluded.csv", "Deprecated outputs excluded from the new report architecture")


Deprecated outputs excluded from the new report architecture: shape=(12, 2); saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_deprecated_outputs_excluded.csv


,Deprecated_Output_Excluded,Reason
0,core_model_selected_threshold_validation_compa...,Historical scope; excluded from approved mappi...
1,core_model_selected_threshold_test_comparison.csv,Historical scope; excluded from approved mappi...
2,core_model_threshold_sensitivity_combined.csv,Historical scope; excluded from approved mappi...
3,report_summary_model_comparison.csv,Historical scope; excluded from approved mappi...
4,report_summary_threshold_selection.csv,Historical scope; excluded from approved mappi...
5,model_comparison_01_selected_test_metrics_bar.png,Historical scope; excluded from approved mappi...
6,model_comparison_02_default_validation_metrics...,Historical scope; excluded from approved mappi...
7,model_comparison_03_selected_validation_metric...,Historical scope; excluded from approved mappi...
8,model_comparison_04_selected_thresholds.png,Historical scope; excluded from approved mappi...
9,model_comparison_05_validation_vs_test_recall_...,Historical scope; excluded from approved mappi...


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_deprecated_outputs_excluded.csv')

## 12. Scope-specific Markdown and LaTeX exports


In [38]:
save_text(report_summary_core.to_latex(index=False, float_format="%.4f"), "report_table_core_research_comparison_latex.tex", "Core RQ2 LaTeX table")
save_text(report_summary_five_model.to_latex(index=False, float_format="%.4f"), "report_table_five_model_default_benchmark_latex.tex", "Five-model benchmark LaTeX table")
save_text(report_summary_boosting_sensitivity.to_latex(index=False, float_format="%.4f"), "report_table_boosting_threshold_sensitivity_latex.tex", "Boosting sensitivity LaTeX table")
save_text(report_summary_boosting_final.to_latex(index=False, float_format="%.4f"), "report_table_boosting_final_test_latex.tex", "Boosting final-test LaTeX table")

core_recall = metric_leader(report_summary_core, "Recall")
benchmark_f1 = metric_leader(report_summary_five_model, "F1")
boosting_recall = metric_leader(report_summary_boosting_final, "Recall")
markdown_lines = [
    "# Report Summary",
    "",
    "## Core Research Comparison (RQ2)",
    f"RQ2 compares Logistic Regression, Random Forest and XGBoost on validation at threshold 0.50. Highest Recall in this scope: {core_recall['Model']} ({core_recall['Recall']:.4f}).",
    "",
    "## Five-Model Default Benchmark",
    f"The additional benchmark compares all five models on validation at threshold 0.50. Highest F1 in this scope: {benchmark_f1['Model']} ({benchmark_f1['F1']:.4f}).",
    "",
    "## Boosting Threshold Sensitivity Analysis",
    "XGBoost, LightGBM and CatBoost only were assessed across validation thresholds. Recall target status and selected thresholds were determined only from validation.",
    "",
    "## Final Boosting Test Results",
    f"Test evaluation used thresholds frozen from validation. Highest test Recall among the three boosting models: {boosting_recall['Model']} ({boosting_recall['Recall']:.4f}). This is metric-specific, not a universal best-model claim.",
    "",
    "## Model Interpretation",
    "Logistic coefficients, Random Forest impurity importance, XGBoost gain, LightGBM gain and CatBoost FeatureImportance are reported separately. Detailed SHAP is XGBoost-only and explains Churn = 1 on the raw margin/log-odds scale.",
    "",
    "## Recommendations",
    "Observed drivers are predictive associations, not causal effects. Operating-model choice should follow the required metric and intervention costs.",
]
save_text("\n".join(markdown_lines), "report_summary_markdown.md", "Scope-specific Markdown report summary")


Core RQ2 LaTeX table: saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_table_core_research_comparison_latex.tex
Five-model benchmark LaTeX table: saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_table_five_model_default_benchmark_latex.tex
Boosting sensitivity LaTeX table: saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_table_boosting_threshold_sensitivity_latex.tex
Boosting final-test LaTeX table: saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_table_boosting_final_test_latex.tex
Scope-specific Markdown report summary: saved=D:\FPTU_Study\SU2026\ADY201m\Project_ADY201m_GR01_AI2009_H1\churn_pipeline_v2\reports\data_reports\report_summary_markdown.md


WindowsPath('D:/FPTU_Study/SU2026/ADY201m/Project_ADY201m_GR01_AI2009_H1/churn_pipeline_v2/reports/data_reports/report_summary_markdown.md')

## 13. Final report validation and source safety


In [39]:
expected_outputs = [
    "report_summary_core_research_comparison.csv",
    "report_summary_five_model_default_benchmark.csv",
    "report_summary_boosting_threshold_sensitivity.csv",
    "report_summary_boosting_final_test.csv",
    "report_summary_model_interpretation_methods.csv",
    "report_summary_metric_specific_recommendations.csv",
    "report_summary_top_churn_drivers.csv",
    "report_summary_business_recommendations.csv",
    "report_summary_deprecated_outputs_excluded.csv",
    "report_figure_manifest.csv",
    "report_table_core_research_comparison_latex.tex",
    "report_table_five_model_default_benchmark_latex.tex",
    "report_table_boosting_threshold_sensitivity_latex.tex",
    "report_table_boosting_final_test_latex.tex",
    "report_table_top_churn_drivers_latex.tex",
    "report_summary_markdown.md",
]
output_check = pd.DataFrame([{"File": name, "Exists": (DATA_REPORTS_DIR / name).exists()} for name in expected_outputs])
display(output_check)
if not output_check["Exists"].all():
    raise FileNotFoundError("Notebook 07 report outputs are incomplete.")

forbidden_selected_models = {"Logistic Regression", "Random Forest"}
for table in [report_summary_boosting_sensitivity, report_summary_boosting_final]:
    if forbidden_selected_models.intersection(set(table["Model"])):
        raise ValueError("Historical core selected-threshold models leaked into boosting report outputs.")

notebook_source = TARGET_NOTEBOOK_PATH.read_text(encoding="utf-8")
forbidden_tokens = ["run" + "_pipeline", ".fi" + "t(", "predict" + "_proba", "import " + "shap"]
found = [token for token in forbidden_tokens if token in notebook_source]
if found:
    raise ValueError(f"Notebook 07 contains forbidden execution tokens: {found}")

print("Notebook 07 completed with explicit RQ2, five-model benchmark, boosting-only sensitivity, frozen-threshold test, and method-separated interpretation scopes.")
print("No training, preprocessing, SHAP computation, database access, SQL execution, or broad selected-output discovery occurs.")


,File,Exists
0,report_summary_core_research_comparison.csv,True
1,report_summary_five_model_default_benchmark.csv,True
2,report_summary_boosting_threshold_sensitivity.csv,True
3,report_summary_boosting_final_test.csv,True
4,report_summary_model_interpretation_methods.csv,True
5,report_summary_metric_specific_recommendations...,True
6,report_summary_top_churn_drivers.csv,True
7,report_summary_business_recommendations.csv,True
8,report_summary_deprecated_outputs_excluded.csv,True
9,report_figure_manifest.csv,True


Notebook 07 completed with explicit RQ2, five-model benchmark, boosting-only sensitivity, frozen-threshold test, and method-separated interpretation scopes.
No training, preprocessing, SHAP computation, database access, SQL execution, or broad selected-output discovery occurs.
